# NB5 · Balance as leg height changes

Compare fixed LQR, frozen-posture gain scheduling, TV-LQR and constrained MPC
on one projected 7 DOF plant. All policies receive the same feasible nominal
trajectory and feedforward. Hip motion is prescribed symmetrically. Required
hip moment, rolling feasibility and wheel moment are checked in each case.
[DUK v3 chapter 07](https://app.notion.com/p/4131f3cc0332817ca50bd61a2dae6b56).

In [ ]:
import hashlib
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from duk_core.modeling import load_leg_geometry_json
from duk_core.visualization.notebook import configure_notebook_plots

configure_notebook_plots()

repo = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "robot/config/physical/duk_leg_geometry.json").is_file()
)
geometry_path = repo / "robot/config/physical/duk_leg_geometry.json"
fixture_path = repo / "research/experiments/duk-v3/synthetic.json"
geometry = load_leg_geometry_json(geometry_path)
fixture = json.loads(fixture_path.read_text())
assert fixture["schema_version"] == 1 and fixture["status"] == "synthetic_research_only"
print(
    "Working tree dirty:",
    bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=repo)),
)
print(
    "Commit",
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip(),
)
for path in (geometry_path, fixture_path):
    print(path.relative_to(repo), hashlib.sha256(path.read_bytes()).hexdigest())
print(fixture["source"])


configure_notebook_plots()

In [ ]:
from duk_core.control.lqr import bryson_weights
from duk_core.modeling import build_planar_biped_model
from duk_core.modeling.hip_load import LinearTorqueSpeed
from duk_core.modeling.planar_numeric import NumericPlanarBiped
from duk_core.modeling.trajectories import HipTrajectory
from duk_core.simulation.height import compare_height_policies

plant = NumericPlanarBiped(build_planar_biped_model(geometry), fixture["planar"])
p = fixture["height_experiment"]
env = fixture["hip_envelope"]
motor = LinearTorqueSpeed(
    env["stall_torque_N_m"], env["no_load_speed_rad_s"], env["efficiency"]
)
v2 = fixture["v2_balance"]
Q, R = bryson_weights(v2["state_scales"], v2["torque_scale_N_m"])
initial, target = np.deg2rad([p["initial_q_deg"], p["final_q_deg"]])
plans = {
    "stationary": HipTrajectory(initial, initial, p["slow_duration_s"]),
    "slow": HipTrajectory(initial, target, p["slow_duration_s"]),
    "fast": HipTrajectory(initial, target, p["fast_duration_s"]),
}
runs = {}
for case, trajectory in plans.items():
    traces = compare_height_policies(
        plant,
        trajectory,
        motor,
        period=p["period_s"],
        duration=p["duration_s"],
        friction=p["friction"],
        motor_bound=p["total_wheel_limit_N_m"],
        q_weight=Q,
        r_weight=R,
        initial_lean=np.deg2rad(p["initial_lean_deg"]),
        horizon=p["mpc_horizon"],
    )
    runs[case] = traces
    for policy, trace in traces.items():
        valid = np.isfinite(trace.applied)
        assert np.all(abs(trace.applied[valid]) <= p["total_wheel_limit_N_m"] + 1e-9)
        assert np.all(trace.grip_usage[valid] <= 1 + 1e-7)
        bounds = np.array(
            [motor.joint_limit(trajectory.evaluate(t)[1]) for t in trace.time]
        )
        assert np.all(
            np.max(abs(trace.hip_effort[valid]), axis=1) <= bounds[valid] + 1e-7
        )
        peak = np.max(abs(np.rad2deg(np.r_[trace.lean, trace.terminal_lean])))
        final = abs(np.rad2deg(trace.terminal_lean))
        passed = (
            valid.all()
            and np.isclose(trace.terminal_time, p["duration_s"])
            and peak <= p["maximum_lean_deg"]
            and final <= p["final_lean_deg"]
        )
        print(
            case,
            policy,
            "meets declared lean criterion:",
            passed,
            "peak/final deg",
            np.round([peak, final], 3),
            "p95 compute ms",
            round(1000 * np.quantile(trace.compute_time, 0.95), 3),
            "terminal time",
            round(trace.terminal_time, 4),
            "statuses",
            sorted(set(trace.status)),
        )
        assert passed, (
            f"Baseline {case}/{policy} failed its declared lean/horizon criterion"
        )

In [ ]:
from duk_core.visualization.notebook import plot_held_signal

fig, axes = plt.subplots(2, 3, figsize=(13, 6), constrained_layout=True)
for column, (case, traces) in enumerate(runs.items()):
    for policy, trace in traces.items():
        axes[0, column].plot(
            np.r_[trace.time, trace.terminal_time],
            np.rad2deg(np.r_[trace.lean, trace.terminal_lean]),
            label=policy,
        )
        axes[1, column].plot(trace.time, trace.grip_usage, label=policy)
    axes[0, column].set(title=case, ylabel="Equivalent lean (deg)")
    axes[1, column].set(xlabel="Time (s)", ylabel="Traction utilization")
    axes[1, column].axhline(1.0, color="0.4", ls=":", label="Traction bound")
    for sign in (-1, 1):
        axes[0, column].axhline(sign * p["maximum_lean_deg"], color="0.4", ls=":")
axes[0, 0].legend()
plt.show()

fast = runs["fast"]["mpc"]
fig, axes = plt.subplots(2, 2, figsize=(10, 6), constrained_layout=True)
axes[0, 0].plot(fast.time, fast.height, label="Body height (m)")
axes[0, 1].plot(
    fast.time,
    [np.rad2deg(plans["fast"].evaluate(t)[0]) for t in fast.time],
    label="q (deg)",
)
plot_held_signal(
    axes[1, 0],
    fast.time,
    fast.requested,
    end_time=fast.terminal_time,
    label="Requested wheel sum",
)
plot_held_signal(
    axes[1, 0],
    fast.time,
    fast.applied,
    end_time=fast.terminal_time,
    label="Applied wheel sum",
)
axes[1, 1].plot(fast.time, fast.hip_effort, label=["Left hip", "Right hip"])
for ax in axes.flat:
    ax.legend()
    ax.set_xlabel("Time (s)")
plt.show()

The comparison is limited to symmetric prescribed motion with ideal sticking
and feasibility checks at command, midpoint and endpoint samples. MPC uses local linear constraints
on the force/effort predictions and a shared applied-torque feasibility
projection. The timing measurements describe this Python run only.

The nominal posture uses analytic first/second derivatives of the composed upper
CoM angle at the current symmetric hip coordinate. The same kinematics supplies
the posture and its derivatives.

**Corrected angle mapping (v3, 2026-11-11).** `trace.state[:, 2]` is lean
relative to the initial fixed pitch offset. `trace.lean`, used in the plots
and criteria, is θ_G0(q) − θ_CCW, the instantaneous equivalent CoM lean.
They coincide only when the upper CoM offset is unchanged. Gain scheduling
alone does not model q̇ and q̈; this common projected plant includes both.


## Compare declared criteria

PASS requires a feasible command for every interval, the complete horizon,
and both declared peak and final lean limits. The final integrated lean is
included. The plots distinguish command samples from the terminal state.
Use the picker to inspect each cached trajectory/policy combination.
The versioned baseline fixture is expected to pass these criteria for all twelve cases. Execution fails if this baseline regresses; stress/fault cases in NB6 retain their explicitly reported failure outcomes.


In [ ]:
from IPython.display import HTML, display

from duk_core.visualization.notebook import (
    case_selector,
    plot_held_signal,
    plot_status_timeline,
    result_table,
)

height_cases = {
    f"{case} / {policy}": (case, trace)
    for case, traces in runs.items()
    for policy, trace in traces.items()
}
rows = []
for name, (case, trace) in height_cases.items():
    peak = np.max(abs(np.rad2deg(np.r_[trace.lean, trace.terminal_lean])))
    final = abs(np.rad2deg(trace.terminal_lean))
    passed = (
        np.isfinite(trace.applied).all()
        and np.isclose(trace.terminal_time, p["duration_s"])
        and peak <= p["maximum_lean_deg"]
        and final <= p["final_lean_deg"]
    )
    rows.append(
        {
            "Case / policy": name,
            "Criterion": "PASS" if passed else "FAIL",
            "End (s)": trace.terminal_time,
            "Peak lean (deg)": peak,
            "Final lean (deg)": final,
            "p95 controller (ms)": 1000 * np.quantile(trace.compute_time, 0.95),
            "Statuses": ", ".join(dict.fromkeys(trace.status)),
        }
    )
display(
    HTML(
        result_table(
            rows,
            caption="NB5 · synthetic plant · peak/final limits "
            f"{p['maximum_lean_deg']:g}° / {p['final_lean_deg']:g}°",
        )
    )
)


def draw_height_case(name):
    case, trace = height_cases[name]
    fig, axes = plt.subplots(
        2, 2, figsize=(11, 6), sharex=True, constrained_layout=True
    )
    axes[0, 0].plot(trace.time, np.rad2deg(trace.lean), label="Command samples")
    axes[0, 0].plot(
        trace.terminal_time,
        np.rad2deg(trace.terminal_lean),
        "o",
        label="Terminal state",
        clip_on=False,
    )
    for sign in (-1, 1):
        axes[0, 0].axhline(sign * p["maximum_lean_deg"], color="0.4", ls=":")
    axes[0, 0].set(
        ylabel="Equivalent lean (deg)", title="Lean and declared peak bounds"
    )
    plot_held_signal(
        axes[0, 1],
        trace.time,
        trace.requested,
        end_time=trace.terminal_time,
        label="Requested",
    )
    plot_held_signal(
        axes[0, 1],
        trace.time,
        trace.applied,
        end_time=trace.terminal_time,
        label="Applied",
    )
    for sign in (-1, 1):
        axes[0, 1].axhline(sign * p["total_wheel_limit_N_m"], color="0.4", ls=":")
    axes[0, 1].set(ylabel="Wheel torque sum (N·m)", title="Held commands")
    axes[1, 0].plot(trace.time, trace.grip_usage, label="Traction utilization")
    axes[1, 0].axhline(1, color="0.4", ls=":", label="Bound")
    axes[1, 0].set(ylabel="Utilization", title="Traction feasibility")
    plot_status_timeline(
        axes[1, 1],
        trace.time,
        trace.status,
        end_time=trace.terminal_time,
        title="Controller / feasibility status",
    )
    for ax in axes.flat:
        ax.set_xlabel("Time (s)")
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
    for ax in axes[0]:
        ax.set_xlabel("")
    fig.suptitle(f"{name} · integrated through {trace.terminal_time:.3f} s")
    return fig


height_picker = case_selector(
    tuple(height_cases), draw_height_case, initial="fast / mpc"
)